# ARK — Colab Training Environment Setup (Phase 6.4)

Installs ARK on a hosted Colab Linux runtime and proves that Python can drive the **real Rapier rover simulation** through the persistent Node.js bridge. No MacBook service required; no PPO training yet (Phase 6.5).

```text
ArkEnv (Gymnasium)  →  JSONL over stdio  →  Node.js bridge (repo-local tsx)
                                              →  ARK RL env  →  Rapier physics
```

Run top to bottom. The only required edit is `REPO_URL` in section 2. Each section fails loudly on error; sections are idempotent and safe to re-run.

## 2. Repository configuration

Set the ARK git URL (required) and an optional revision. A pinned commit SHA gives reproducible experiments. For private repos, store the URL — including a token, `https://<token>@github.com/<org>/<repo>.git` — in a Colab secret and read it with `userdata.get`; it is never printed by any cell.

In [ ]:
import os

REPO_URL = ""  # REQUIRED, e.g. "https://github.com/<org>/ark.git"
# Private-repo alternative (Colab secret, never printed):
# from google.colab import userdata
# REPO_URL = userdata.get("ARK_REPO_URL")
ARK_REV = ""  # branch, tag, or commit SHA; empty = remote default branch
WORK_DIR = "/content/ARK"
OUTPUT_DIR = "/content/ARK/colab_artifacts"
MOUNT_DRIVE = False  # set True to copy artifacts to Google Drive (asks for auth)
SEED = 42

if not REPO_URL or not REPO_URL.startswith(("https://", "git@")):
    raise ValueError("Set REPO_URL to the ARK git URL before continuing.")
os.environ["ARK_REPO_URL"] = REPO_URL
os.environ["ARK_REV"] = ARK_REV
os.environ["ARK_WORK_DIR"] = WORK_DIR
print("config OK — work dir:", WORK_DIR, "| rev:", ARK_REV or "(default branch)")

In [ ]:
import subprocess

def sh(args):
    try:
        r = subprocess.run(args, capture_output=True, text=True, check=True, timeout=600)
        return r.stdout.strip()
    except subprocess.CalledProcessError as e:
        # Never leak credentials that may be embedded in the URL.
        raise RuntimeError(f"git failed: {(e.stderr or '')[-500:].replace(REPO_URL, '<redacted>')}")

if os.path.isdir(os.path.join(WORK_DIR, ".git")):
    print("reusing existing checkout")
    sh(["git", "-C", WORK_DIR, "fetch", "--tags", "origin"])
else:
    print("cloning (URL redacted from output)")
    os.makedirs(os.path.dirname(WORK_DIR), exist_ok=True)
    sh(["git", "clone", REPO_URL, WORK_DIR])
if ARK_REV:
    sh(["git", "-C", WORK_DIR, "checkout", ARK_REV])
# Some remotes nest the project (clone/ARK/...); descend when needed.
if not os.path.isfile(os.path.join(WORK_DIR, "services/training/scripts/setup_colab.sh")):
    nested = os.path.join(WORK_DIR, "ARK")
    if os.path.isfile(os.path.join(nested, "services/training/scripts/setup_colab.sh")):
        WORK_DIR = nested
        OUTPUT_DIR = os.path.join(WORK_DIR, "colab_artifacts")
        os.environ["ARK_WORK_DIR"] = WORK_DIR
        print("nested layout detected — work dir:", WORK_DIR)
assert os.path.isfile(os.path.join(WORK_DIR, "services/training/scripts/setup_colab.sh")), \
    "ARK training scripts missing — checkout predates Phase 6.3; push local work first."
print("commit:", sh(["git", "-C", WORK_DIR, "rev-parse", "HEAD"]))

## 3. Runtime diagnostics

Records the hosted runtime. A GPU is optional: Rapier physics runs on CPU, and small vector-policy PPO trains fine on CPU initially.

In [ ]:
import platform, shutil, subprocess

print("python:", platform.python_version())
print("platform:", platform.platform())
print("cpus:", os.cpu_count())
try:
    mem_kb = [l for l in open("/proc/meminfo") if l.startswith("MemTotal:")][0].split()[1]
    print("memory:", f"{int(mem_kb) / 1e6:.1f} GB")
except Exception as e:
    print("memory: unknown", e)

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip())
else:
    print("gpu: none reported (OK — physics runs on CPU)")

try:
    import torch
    print("torch:", torch.__version__, "| cuda_available:", torch.cuda.is_available())
except ImportError:
    print("torch: not installed (optional; needed only for Phase 6.5 training)")

## 4. Node.js setup

Requires Node.js >= 18 (the repo declares no `engines` floor; Rapier WASM + tsx are verified on 18+). An older runtime is upgraded to Node 22 LTS; a compatible runtime is left untouched.

In [ ]:
subprocess.run(["bash", f"{WORK_DIR}/services/training/scripts/setup_colab.sh", "node"], check=True)
v = subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip()
assert int(v.lstrip("v").split(".")[0]) >= 18, v
print("node OK:", v)

## 5. Python dependency setup

Installs into the notebook kernel's own Python (uv venvs are invisible to Colab cells). Gymnasium is pinned from `uv.lock`; NumPy floats at `>=1.26` so Colab's preinstalled build is kept when compatible. Exact versions are recorded in the artifacts.

In [ ]:
subprocess.run(["bash", f"{WORK_DIR}/services/training/scripts/setup_colab.sh", "python"],
                 check=True, env={**os.environ, "ARK_COLAB": "1"})
import ark_gym, gymnasium, numpy
print("ark_gym", ark_gym.__version__, "| gymnasium", gymnasium.__version__, "| numpy", numpy.__version__)

## 6. Build and installation

Reproducible workspace install via `npm ci` (lockfile). There is no TypeScript build step: the bridge runs directly through the repo-local `tsx`.

In [ ]:
subprocess.run(["bash", f"{WORK_DIR}/services/training/scripts/setup_colab.sh", "npm"], check=True)
assert os.path.isfile(f"{WORK_DIR}/node_modules/.bin/tsx"), "tsx missing after npm ci"
rapier = subprocess.run(["node", "-p", "require('./node_modules/@dimforge/rapier3d-compat/package.json').version"],
                         cwd=WORK_DIR, capture_output=True, text=True).stdout.strip()
print("tsx OK | rapier3d-compat", rapier)

## 7. Bridge initialization

Records versions, then opens the persistent Node.js bridge directly to prove the JSONL transport and live space contracts.

In [ ]:
subprocess.run(["python", f"{WORK_DIR}/services/training/scripts/verify_colab.py",
                "--stage", "versions", "--output-dir", OUTPUT_DIR], check=True)
from ark_gym import BridgeClient
with BridgeClient() as client:
    hello = client.hello()
    assert hello["protocolVersion"] == "1.0", hello
    spaces = client.get_spaces()
    print("hello:", hello)
    print("observation shape:", spaces["observation"]["shape"], "| action shape:", spaces["action"]["shape"])
print("Bridge OK: persistent Node.js process responding over JSONL.")

## 8. Earth environment smoke test

Real Earth episode: planet identity is asserted from simulation metadata, and forward driving must produce measured Rapier motion (position deltas, energy burn) — never mock observations.

In [ ]:
subprocess.run(["python", f"{WORK_DIR}/services/training/scripts/verify_colab.py",
                "--stage", "planets", "--only-planet", "earth", "--seed", str(SEED),
                "--output-dir", OUTPUT_DIR], check=True)

## 9. Mars environment smoke test

Same protocol on Mars (lower gravity → visibly less traction for identical drive effort).

In [ ]:
subprocess.run(["python", f"{WORK_DIR}/services/training/scripts/verify_colab.py",
                "--stage", "planets", "--only-planet", "mars", "--seed", str(SEED),
                "--output-dir", OUTPUT_DIR], check=True)

## 10. Gymnasium validation

Runs Gymnasium's environment checker (spaces, dtypes, flags, reset semantics, determinism). Stable-Baselines3's checker runs automatically if SB3 is installed; installing it here is optional and slow (PyTorch).

In [ ]:
# Optional (slow, downloads PyTorch): also run SB3's checker.
# !pip install -q stable-baselines3
subprocess.run(["python", f"{WORK_DIR}/services/training/scripts/verify_colab.py",
                "--stage", "gym", "--output-dir", OUTPUT_DIR], check=True)

## 11. Performance benchmark

Bounded throughput run (warmup + 300 timed RL steps): steps/s, reset cost, CPU/memory. All values measured, none invented.

In [ ]:
subprocess.run(["python", f"{WORK_DIR}/services/training/scripts/verify_colab.py",
                "--stage", "benchmark", "--steps", "300", "--seed", str(SEED),
                "--output-dir", OUTPUT_DIR], check=True)

## 12. Save diagnostics

Finalizes `ark_colab_diagnostics.json` (versions, commit, spaces, benchmark, scenario config — never credentials). Drive copy is optional and gated by `MOUNT_DRIVE`; the next cell downloads the file as a fallback.

In [ ]:
import json
subprocess.run(["python", f"{WORK_DIR}/services/training/scripts/verify_colab.py",
                "--stage", "artifacts", "--output-dir", OUTPUT_DIR], check=True)
diag_path = f"{OUTPUT_DIR}/ark_colab_diagnostics.json"
print(open(diag_path).read()[:1500], "\n...")
if MOUNT_DRIVE:
    import shutil, time
    from google.colab import drive
    drive.mount("/content/drive")
    dest = f"/content/drive/MyDrive/ark_colab_{time.strftime('%Y%m%d_%H%M%S')}"
    shutil.copytree(OUTPUT_DIR, dest)
    print("copied to", dest)
else:
    print("MOUNT_DRIVE=False — artifacts kept local. Run the next cell to download.")

In [ ]:
try:
    from google.colab import files
    files.download(f"{OUTPUT_DIR}/ark_colab_diagnostics.json")
except ImportError:
    print("Not on Colab; artifact at:", f"{OUTPUT_DIR}/ark_colab_diagnostics.json")

## 13. Troubleshooting

- `Set REPO_URL ...` — fill in the repository URL in section 2 and re-run from there.
- `git clone/failed` — wrong URL or missing credentials; for private repos use a token URL from a Colab secret.
- `tsx missing` / bridge `Cannot find tsx` — re-run section 6 (`npm ci`); check the npm log for network errors.
- `No bridge response within ...s` — the Node process is wedged; Runtime → Restart runtime and re-run top to bottom (orphaned `tsx` processes die with the runtime).
- `pip install` NumPy conflicts — the setup keeps Colab's NumPy when it satisfies `>=1.26`; the recorded version in the artifacts is authoritative.
- Slow first run — `npm ci` and the first Rapier WASM init dominate; re-runs reuse the checkout and installed packages.
- `Not an ARK checkout` — `WORK_DIR` points at the wrong directory; verify the clone cell output.
- `training scripts missing` — the checkout predates Phase 6.3/6.4; commit and push local work, then re-run.

## 14. Readiness report

Verdict for Phase 6.5 (PPO), derived from the recorded artifacts — not from printed labels.

In [ ]:
d = json.load(open(f"{OUTPUT_DIR}/ark_colab_diagnostics.json"))
missing = [k for k in ["versions", "planets", "gym", "benchmark", "saved_at"] if k not in d]
assert not missing, f"missing stages: {missing} — re-run the notebook top to bottom"
assert set(d["planets"]) >= {"earth", "mars"}, d["planets"].keys()
b, v = d["benchmark"], d["versions"]
print("ARK Colab setup: READY FOR PHASE 6.5 (PPO)")
print(f"- commit: {v['repo_commit'][:12]} | node {v['node']} | rapier {v['rapier3d-compat']}")
print(f"- python {v['python']} | gymnasium {v['gymnasium']} | numpy {v['numpy']}")
print(f"- earth traveled {d['planets']['earth']['traveled_m']:.1f}m | mars traveled {d['planets']['mars']['traveled_m']:.1f}m (real Rapier motion)")
print(f"- gymnasium check_env: {d['gym']['gymnasium_check_env']} | sb3: {d['gym']['sb3_check_env']}")
print(f"- throughput: {b['rl_steps_per_s']} RL steps/s | reset {b['avg_reset_ms']}ms | cpu: {b['cpu']}")